# mixed4e:55 reproduction

Runs the pipeline step by step with the `neural_interp` package on a Colab GPU. Method: [method.md](../../thesis/method.md). Notes and results: [README.md](README.md).

In [ ]:
REF = "main"
!pip install -q "neural-interp @ git+https://github.com/priyanshujain/neural-interp@{REF}"

## Data

In [ ]:
from google.colab import auth
auth.authenticate_user()
!mkdir -p /content/data /content/reference
!test -d /content/data/inception_pool || (gcloud storage cp gs://pj-okc/inception_pool.tar /content/ && tar --warning=no-unknown-keyword -xf /content/inception_pool.tar -C /content/data && find /content/data -name '._*' -delete)
!test -d /content/reference/runs || (gcloud storage cp gs://pj-okc/mixed4e_55/mixed4e_55_full.tar /content/ && tar -xf /content/mixed4e_55_full.tar -C /content/reference runs/mixed4e_55/space)

## Run config

`same_images` uses the reference run's 2,000 images; `fresh_sample` draws 2,000 with `SEED`.

In [ ]:
import json
from pathlib import Path

import numpy as np

from neural_interp.buckets import cluster_buckets, set_buckets
from neural_interp.density_peaks import build_forest, collapse_copies
from neural_interp.distance import weighted_jaccard
from neural_interp.experiments.compare import compare
from neural_interp.experiments.mixed4e_55 import composition, device, hit_id, wnid
from neural_interp.layout import hit_map, map_nodes
from neural_interp.membership import soft_membership
from neural_interp.report import Tile, write_report
from neural_interp.selection import elbow_hits
from neural_interp.space import HitSpace, summary
from neural_interp.vision import inception
from neural_interp.vision.cards import attribution_overlay, framed, save_tiles
from neural_interp.vision.labels import group, synset_name

RUN = "same_images"
SEED = 1
N_IMAGES = 2000
K = 5
PEAK_SHARE = 0.8
MIN_BUCKET = 10

POOL = Path("/content/data/inception_pool")
REFERENCE = Path("/content/reference/runs/mixed4e_55/space")
OUT = Path("/content/runs") / RUN
SPACE = OUT / "space"
DEV = device()
DEV

## Model and images

In [ ]:
model = inception.load_model(DEV)
probe = inception.Probe(model, inception.MIXED4E_55)
if RUN == "same_images":
    ids = np.load(REFERENCE / "hits.npz", allow_pickle=True)["sample_ids"]
    paths = [POOL / i for i in ids]
else:
    paths = inception.sample_images(POOL, N_IMAGES, SEED)
len(paths), paths[0]

## Activations

In [ ]:
activations = inception.collect_activations(probe, paths, DEV)
grid = activations.shape[1:]
flat = activations.ravel()
activations.shape, flat.max(), (flat > 0).mean()

## Hits (elbow)

In [ ]:
hits = elbow_hits(flat)
len(hits), flat[hits[-1]]

## Hit vectors

In [ ]:
rows, preactivation = inception.collect_hadamard(probe, paths, hits, grid, DEV)
rows.shape, np.abs(rows.sum(1) - preactivation).max()

## Copies and distances

In [ ]:
unique, counts, point = collapse_copies(rows)
dist = weighted_jaccard(unique)
len(unique), counts.max(), dist.shape

## Density-peak forest

In [ ]:
forest = build_forest(dist, counts, K)
len(forest.roots), forest.rho_cut, forest.delta_cut, np.bincount(forest.tree, weights=counts).astype(int)

## Soft membership

In [ ]:
membership = soft_membership(dist, forest, K, PEAK_SHARE)
membership.temperature, [len(p) for p in membership.peaks], len(membership.touching)

## Save the hit space

In [ ]:
space = HitSpace([hit_id(paths, POOL, h, grid) for h in hits], flat[hits], rows, point, counts, dist, forest, membership)
space.save(SPACE)
np.save(SPACE / "all_activations.npy", flat)
(SPACE / "summary.json").write_text(json.dumps(summary(space), indent=1))
{k: v for k, v in summary(space).items() if k != "cluster_sizes"}

## Composition

In [ ]:
comp = composition(space)
comp[:10]

## Buckets and map

In [ ]:
buckets = [b for b in cluster_buckets(space.probs) + set_buckets(space.probs) if len(b.hits) >= MIN_BUCKET]
root_hits = np.array([int(np.flatnonzero(space.point == r)[0]) for r in space.forest.roots])
nodes = map_nodes(buckets, space.probs, root_hits)
positions = hit_map(np.asarray(space.dist)[np.ix_(space.point, space.point)], space.probs, nodes, root_hits)
len(buckets), len(nodes)

## Report

In [ ]:
cards = OUT / "report" / "cards"
deeplift = inception.NeuronAttribution(model, inception.MIXED4E_55)

def tile(hit: int) -> Tile:
    path, pos = space.hit_ids[hit].rsplit(":", 1)
    y, x = divmod(int(pos), grid[1])
    pixels = inception.preprocess(POOL / path)
    attribution = deeplift.map(pixels.to(DEV), y, x)
    array = pixels.numpy()
    image, overlay = save_tiles(cards, f"hit_{hit}", framed(array, y, x, grid[1]),
                                attribution_overlay(array, attribution, y, x, grid[1]))
    return Tile(image, overlay, synset_name(wnid(space.hit_ids[hit])))

write_report(OUT / "report", space, buckets, positions, tile,
             group_of=lambda h: group(wnid(space.hit_ids[h])), all_activations=flat, unit="InceptionV1 mixed4e:55")
(OUT / "report" / "composition.json").write_text(json.dumps(comp, indent=1))

## Compare with the reference

In [ ]:
result = compare(SPACE, REFERENCE, OUT / "compare.json")
print("shared hits", result["shared_hits"], "ARI", result["ari_on_shared"])
for side in ["ours", "reference"]:
    w = result[side]
    print(side, w["hits"], "hits", w["clusters"], "clusters", f"purity {w['weighted_purity']:.2f}",
          {k: round(v, 2) for k, v in w["group_share"].items()}, "cat+vehicle mixed", w["clusters_mixing_cat_and_vehicle"])

## Upload

In [ ]:
!gcloud storage cp -r {OUT} gs://pj-okc/neural_interp/mixed4e_55/